**Auteur(s)** : Cheikhou Akhmed KANE

**Description** : Ajout des donnees sols (littérature et dire d'experts) pour le fichier `typeDeSolParZH.shp`

## 1. Description du projet

Ce notebook est une étape intermédiaire visant à enrichir notre tableau de synthèse avec des données qui ne sont pas disponibles dans les bases de données géospatiales. Il s'agit d'intégrer des valeurs issues de la **littérature scientifique** (thèses, publications) et de **dire d'experts**.

---
## 2. Objectifs

* Charger la table de synthèse des sols créée par le notebook `02b`.
* Compléter les valeurs manquantes pour les **variables globales** (`PIRM`, `PRO`, `CSTRU`).
* Compléter les valeurs manquantes pour les **variables par couche** (`KSAT`, etc.).
* **Ajouter les colonnes d'identifiants** (`ID_ZH`, `STU_DOM` et `ID_SOL`) requises par MAELIA.
* Sauvegarder la table de synthèse finale et complète.

---
## 3. Sources des données

### 3.1. Identifiants

| Variable | Description | Source |
| :--- | :--- | :--- |
| `ID_ZH` | Identifiant de la Zone Hydrographique. | Défini dans ce notebook (`'SSM1'`). |
| `STU_DOM` | Unité Typologique de Sol Dominante. | Défini dans ce notebook ('sableux'/'argileux'). |
| `ID_SOL` | Identifiant unique de l'unité de sol. | Créé dans ce notebook (combinaison des autres ID). |

### 3.2. Variables Globales

| Variable | Description | Source |
| :--- | :--- | :--- |
| `PIRM` | Infiltrabilité du sol (mm/h) | Thèse de Waly Faye & Hypothèses |
| `PRO` | Profondeur totale du sol (cm) | Hypothèse de modélisation (60 cm) |
| `CSTRU` | Qualité de la structure du sol (note) | Dire d'expert (0.5) |

### 3.3. Variables par Couche

| Variable | Description | Source |
| :--- | :--- | :--- |
| `P1`, `P2` | Profondeur cumulative des horizons (cm) | Hypothèse de modélisation (30, 60 cm) |
| `KSAT1`, `KSAT2` | Conductivité hydraulique à saturation (mm/h) | Thèse de Waly Faye & Hypothèses |
| `EG1`, `EG2` | Teneur en éléments grossiers (%) | Dire d'expert (0) |
| `CAL1`, `CAL2` | Teneur en calcaire (%) | Dire d'expert (0) |

---
## 4. Méthodologie d'Estimation des Données Manquantes

### 4.1. PIRM : Infiltrabilité du sol (mm/h)

Les données de base pour la variable `PIRM` sont issues de la **thèse de Waly Faye**, qui a mesuré cette propriété pour deux des types d'îlots de notre zone d'étude. Pour compléter les six types d'îlots manquants, nous avons formulé des **hypothèses basées sur la logique agronomique** :

1.  **Effet "Arbre"** : Les données mesurées montrent que la présence d'arbres augmente l'infiltrabilité d'environ 21% sur les sols Dior.
2.  **Effet "Champ de Case"** : Nous avons posé l'hypothèse qu'un champ de case, plus travaillé, a une infiltrabilité au moins aussi bonne qu'un champ de brousse avec arbres.
3.  **Effet "Type de Sol"** : Les sols "Deck", plus argileux, ont une infiltrabilité structurellement plus faible. Nous avons estimé leur `PIRM` à 50% de celle des sols "Dior" dans des conditions équivalentes.

In [13]:
import pandas as pd
from pathlib import Path
import numpy as np

In [2]:
base_dir = Path.cwd().parent.resolve()

# Fichier en entrée (la table de synthèse agrégée du notebook 02b)
input_csv_path = base_dir / "data" / "sols" / "csv" / "processed" / "donnees_typesDeSol.csv"

# Fichier en sortie (la même table, mais enrichie avec les données expertes)
output_csv_path = base_dir / "data" / "sols" / "csv" / "processed" / "donnees_typesDeSol_enrichies.csv"

In [3]:
# --- CHARGEMENT --- #
try:
    # On utilise sep=';' car c'est le séparateur que nous avons utilisé à la sauvegarde
    df_synthese = pd.read_csv(input_csv_path, sep=';')
    
    print("✅ Table de synthèse chargée avec succès.")
    print(f"   -> Contient {df_synthese.shape[0]} lignes et {df_synthese.shape[1]} colonnes.")
    display(df_synthese.head())

except FileNotFoundError:
    print(f"🚨 ERREUR : Fichier non trouvé. Vérifiez le chemin : {input_csv_path}")

✅ Table de synthèse chargée avec succès.
   -> Contient 8 lignes et 23 colonnes.


,ZONE_PEDO,ARG1,ARG2,SAB1,SAB2,DAH1,DAH2,C1,C2,MO1,...,PH1,PH2,HCC1,HCC2,HPFP1,HPFP2,RUPRH1,RUPRH2,CN1,CN2
0,dekk/mbel_cb_avec_arbr,18.066667,18.800000,64.400000,64.733333,1.557333,1.514667,0.468667,0.323333,0.807981,...,5.864444,5.806667,19.926366,20.082567,9.113581,9.235677,32.438357,32.540670,13.107424,10.096399
1,dekk/mbel_cb_sans_arbr,18.100000,18.700000,64.450000,64.850000,1.556500,1.509000,0.474500,0.325500,0.818038,...,5.868421,5.815789,19.618432,20.266224,8.976415,9.353365,31.926052,32.738577,13.494923,10.308826
2,dekk_cb_avec_arbr,17.492063,18.365079,66.047619,65.984127,1.546032,1.508889,0.436984,0.303810,0.753361,...,5.841799,5.766667,18.820873,19.060625,8.657769,8.797480,30.489314,30.789434,13.057301,9.962351
3,dekk_cb_sans_arbr,17.457831,18.385542,66.060241,65.951807,1.545422,1.509639,0.433373,0.302530,0.747136,...,5.827642,5.763415,19.107418,19.225314,8.763053,8.867642,31.033094,31.073014,13.124069,10.012450
4,dior_cb_avec_arbr,17.647668,18.481865,65.860104,65.678756,1.547565,1.507409,0.450725,0.311140,0.777051,...,5.839062,5.776166,18.849091,19.341612,8.677742,8.947725,30.514047,31.181659,13.723203,10.246329


In [4]:
# --- 4.1 Ajout de la variable PIRM (Infiltrabilité) ---

# 1. Créer le dictionnaire de correspondance basé sur nos hypothèses
pirm_map = {
    'dior_cb_avec_arbr': 840.48,
    'dior_cb_sans_arbr': 696.00,
    'dior_cc_avec_arbr': 840.48,      # Hypothèse: cc_avec_arbr ≈ cb_avec_arbr
    'dior_cc_sans_arbr': 696.00,      # Hypothèse: cc_sans_arbr ≈ cb_sans_arbr
    'dekk_cb_avec_arbr': 420.24,      # Hypothèse: dekk ≈ 50% de dior
    'dekk_cb_sans_arbr': 348.00,      # Hypothèse: dekk ≈ 50% de dior
    'dekk/mbel_cb_avec_arbr': 420.24, # Hypothèse: dekk/mbel ≈ dekk
    'dekk/mbel_cb_sans_arbr': 348.00  # Hypothèse: dekk/mbel ≈ dekk
}

# 2. Appliquer le mapping pour créer la nouvelle colonne
df_synthese['PIRM'] = df_synthese['ZONE_PEDO'].map(pirm_map)

# --- VÉRIFICATION ---
print("✅ Colonne 'PIRM' ajoutée avec succès.")
print(f"   Dimensions actuelles : {df_synthese.shape[0]} lignes et {df_synthese.shape[1]} colonnes.")
print("\nAperçu des valeurs de PIRM :")

# Afficher toutes les lignes pour vérifier chaque ZONE_PEDO
display(df_synthese[['ZONE_PEDO', 'PIRM']])

✅ Colonne 'PIRM' ajoutée avec succès.
   Dimensions actuelles : 8 lignes et 24 colonnes.

Aperçu des valeurs de PIRM :


,ZONE_PEDO,PIRM
0,dekk/mbel_cb_avec_arbr,420.24
1,dekk/mbel_cb_sans_arbr,348.00
2,dekk_cb_avec_arbr,420.24
3,dekk_cb_sans_arbr,348.00
4,dior_cb_avec_arbr,840.48
5,dior_cb_sans_arbr,696.00
6,dior_cc_avec_arbr,840.48
7,dior_cc_sans_arbr,696.00


## 4.2. KSAT : Conductivité hydraulique à saturation (mm/h)

La conductivité hydraulique à saturation (`KSAT`) est également issue de la **thèse de Waly Faye**. Les données disponibles ne sont pas fournies par couche de sol mais pour le profil global. De plus, les valeurs ne sont disponibles que pour 4 des 8 types d'îlots.

Pour compléter les données manquantes, les hypothèses suivantes ont été posées.

### Hypothèses d'estimation

1.  **Effet "Champ de Case" (cc)** : On suppose qu'il a un comportement similaire au champ de brousse dans des conditions identiques. Exemple : `KSAT_dior_cc ≈ KSAT_dior_cb`.
2.  **Effet "Deck/Mbel"** : On suppose un comportement identique au sol "Deck". Exemple : `KSAT_dekk/mbel ≈ KSAT_dekk`.
3.  **Répartition par couche** : La `KSAT` est généralement plus élevée en surface. On pose l'hypothèse que la valeur mesurée est la plus représentative de la couche de surface (**`KSAT1`**), et que la conductivité de la couche inférieure (**`KSAT2`**) est **75% de `KSAT1`**.

### Tableau des estimations finales

| ZONE_PEDO | KSAT1 (mm/h) | KSAT2 (mm/h) |
| :--- | :--- | :--- |
| `dior_cb_avec_arbr` | **674.64** | **505.98** |
| `dior_cb_sans_arbr` | **480.24** | **360.18** |
| `dekk_cb_avec_arbr` | **322.74** | **242.06** |
| `dekk_cb_sans_arbr` | **296.28** | **222.21** |
| `dior_cc_avec_arbr` | **674.64** | **505.98** |
| `dior_cc_sans_arbr` | **480.24** | **360.18** |
| `dekk/mbel_cb_avec_arbr`| **322.74**| **242.06** |
| `dekk/mbel_cb_sans_arbr`| **296.28**| **222.21** |

In [5]:
# --- Ajout des variables KSAT1 et KSAT2 ---

# 1. Créer le dictionnaire de correspondance pour la valeur de profil
ksat_profil_map = {
    'dior_cb_avec_arbr': 674.64,
    'dior_cb_sans_arbr': 480.24,
    'dekk_cb_avec_arbr': 322.74,
    'dekk_cb_sans_arbr': 296.28,
    'dior_cc_avec_arbr': 674.64,      # Hypothèse
    'dior_cc_sans_arbr': 480.24,      # Hypothèse
    'dekk/mbel_cb_avec_arbr': 322.74, # Hypothèse
    'dekk/mbel_cb_sans_arbr': 296.28  # Hypothèse
}

# 2. Créer une colonne temporaire avec la valeur de profil
df_synthese['ksat_profil'] = df_synthese['ZONE_PEDO'].map(ksat_profil_map)

# 3. Appliquer l'hypothèse de répartition pour créer KSAT1 et KSAT2
# KSAT1 prend la valeur du profil
df_synthese['KSAT1'] = df_synthese['ksat_profil']
# KSAT2 prend 75% de la valeur de KSAT1
df_synthese['KSAT2'] = df_synthese['ksat_profil'] * 0.75

# 4. Supprimer la colonne de travail
df_synthese.drop(columns=['ksat_profil'], inplace=True)

# --- VÉRIFICATION ---
print("✅ Colonnes 'KSAT1' et 'KSAT2' ajoutées avec succès.")
print("\nAperçu des valeurs de KSAT :")
display(df_synthese[['ZONE_PEDO', 'KSAT1', 'KSAT2']])

✅ Colonnes 'KSAT1' et 'KSAT2' ajoutées avec succès.

Aperçu des valeurs de KSAT :


,ZONE_PEDO,KSAT1,KSAT2
0,dekk/mbel_cb_avec_arbr,322.74,242.055
1,dekk/mbel_cb_sans_arbr,296.28,222.210
2,dekk_cb_avec_arbr,322.74,242.055
3,dekk_cb_sans_arbr,296.28,222.210
4,dior_cb_avec_arbr,674.64,505.980
5,dior_cb_sans_arbr,480.24,360.180
6,dior_cc_avec_arbr,674.64,505.980
7,dior_cc_sans_arbr,480.24,360.180


### 4.3. PRO : Profondeur totale du sol (cm)

La profondeur totale du sol explorable par les racines (`PRO`) a été fixée à **60 cm** pour l'ensemble des types d'îlots.

Ce choix est guidé par la disponibilité des données sources, qui ont été harmonisées sur deux horizons de 30 cm chacun (0-30 cm et 30-60 cm), définissant ainsi la profondeur totale de notre profil de sol modélisé.

In [6]:
# --- 4.3 Ajout de la variable PRO (Profondeur) ---

# Définir la valeur fixe
profondeur_sol = 60  # en cm

# Ajouter la colonne avec cette valeur fixe
df_synthese['PRO'] = profondeur_sol

# --- VÉRIFICATION ---
print("✅ Colonne 'PRO' ajoutée avec succès.")
display(df_synthese[['ZONE_PEDO', 'PRO']].head())

✅ Colonne 'PRO' ajoutée avec succès.


,ZONE_PEDO,PRO
0,dekk/mbel_cb_avec_arbr,60
1,dekk/mbel_cb_sans_arbr,60
2,dekk_cb_avec_arbr,60
3,dekk_cb_sans_arbr,60
4,dior_cb_avec_arbr,60


### 4.4. CSTRU : Qualité de la Structure du Sol

La variable `CSTRU` est une **note experte** comprise entre 0 et 1 qui représente la qualité structurale globale du sol.

Pour cette première version de l'instanciation, une valeur de **0.5** a été assignée à l'ensemble des sols.

In [7]:
# --- 4.4 Ajout de la variable CSTRU (Structure) ---

# Définir la valeur fixe
structure_sol = 0.5  # note experte

# Ajouter la colonne avec cette valeur fixe
df_synthese['CSTRU'] = structure_sol

# --- VÉRIFICATION ---
print("✅ Colonne 'CSTRU' ajoutée avec succès.")
display(df_synthese[['ZONE_PEDO', 'CSTRU']].head())

✅ Colonne 'CSTRU' ajoutée avec succès.


,ZONE_PEDO,CSTRU
0,dekk/mbel_cb_avec_arbr,0.5
1,dekk/mbel_cb_sans_arbr,0.5
2,dekk_cb_avec_arbr,0.5
3,dekk_cb_sans_arbr,0.5
4,dior_cb_avec_arbr,0.5


### 4.5. P : Profondeur des Horizons (cm)

Le profil de sol est modélisé avec **deux horizons**. Conformément aux exigences de MAELIA, les variables `P1` et `P2` représentent la **profondeur cumulative** (ou "emboîtée") depuis la surface.

* **`P1`** est fixée à **30 cm**, représentant la profondeur de la première couche (0-30 cm).
* **`P2`** est fixée à **60 cm**, représentant la profondeur totale du profil étudié (0-60 cm).

In [8]:
# --- 4.5 Ajout des variables P1 et P2 (Profondeur) ---

# Définir les valeurs fixes de profondeur cumulative
profondeur_h1 = 30  # en cm
profondeur_h2 = 60  # en cm

# Ajouter les colonnes avec ces valeurs fixes
df_synthese['P1'] = profondeur_h1
df_synthese['P2'] = profondeur_h2

# --- VÉRIFICATION ---
print("✅ Colonnes 'P1' et 'P2' ajoutées avec succès.")
display(df_synthese[['ZONE_PEDO', 'P1', 'P2']].head())

✅ Colonnes 'P1' et 'P2' ajoutées avec succès.


,ZONE_PEDO,P1,P2
0,dekk/mbel_cb_avec_arbr,30,60
1,dekk/mbel_cb_sans_arbr,30,60
2,dekk_cb_avec_arbr,30,60
3,dekk_cb_sans_arbr,30,60
4,dior_cb_avec_arbr,30,60


### 4.6. EG : Teneur en Éléments Grossiers (%)

La teneur en éléments grossiers (`EG`) représente le pourcentage de fragments dont le diametre est superieur a 2mm.

En se basant sur le **dire d'experts**, cette valeur est considérée comme nulle pour notre zone d'étude. Les variables `EG1` et `EG2` sont donc fixées à **0**.

In [9]:
# --- 4.6 Ajout des variables EG1 et EG2 (Éléments Grossiers) ---

# Définir la valeur fixe
elements_grossiers = 0  # en %

# Ajouter les colonnes avec cette valeur fixe
df_synthese['EG1'] = elements_grossiers
df_synthese['EG2'] = elements_grossiers

# --- VÉRIFICATION ---
print("✅ Colonnes 'EG1' et 'EG2' ajoutées avec succès.")
display(df_synthese[['ZONE_PEDO', 'EG1', 'EG2']].head())

✅ Colonnes 'EG1' et 'EG2' ajoutées avec succès.


,ZONE_PEDO,EG1,EG2
0,dekk/mbel_cb_avec_arbr,0,0
1,dekk/mbel_cb_sans_arbr,0,0
2,dekk_cb_avec_arbr,0,0
3,dekk_cb_sans_arbr,0,0
4,dior_cb_avec_arbr,0,0


### 4.7. CAL : Teneur en Calcaire (%)

La teneur en calcaire (`CAL`) est le pourcentage de carbonate de calcium dans le sol.

En se basant sur le **dire d'experts**, cette valeur est considérée comme nulle pour les sols de la zone. Les variables `CAL1` et `CAL2` sont donc fixées à **0**.

In [10]:
# --- 4.7 Ajout des variables CAL1 et CAL2 (Calcaire) ---

# Définir la valeur fixe
teneur_calcaire = 0  # en %

# Ajouter les colonnes avec cette valeur fixe
df_synthese['CAL1'] = teneur_calcaire
df_synthese['CAL2'] = teneur_calcaire

# --- VÉRIFICATION ---
print("✅ Colonnes 'CAL1' et 'CAL2' ajoutées avec succès.")
display(df_synthese[['ZONE_PEDO', 'CAL1', 'CAL2']].head())

✅ Colonnes 'CAL1' et 'CAL2' ajoutées avec succès.


,ZONE_PEDO,CAL1,CAL2
0,dekk/mbel_cb_avec_arbr,0,0
1,dekk/mbel_cb_sans_arbr,0,0
2,dekk_cb_avec_arbr,0,0
3,dekk_cb_sans_arbr,0,0
4,dior_cb_avec_arbr,0,0


### 4.8. ID_ZH : Identifiant de la Zone Hydrographique

Pour cette étude, l'ensemble du parcellaire est considéré comme appartenant à une seule et même **Zone Hydrographique (ZH)**.

L'identifiant unique **`'SSM1'`** (pour Sasseme 1) a été assigné à toutes les unités de sol.

In [11]:
# --- 4.8 Ajout de la variable ID_ZH ---

# Définir la valeur fixe
id_zone_hydro = 'SSM1'

# Ajouter la colonne avec cette valeur fixe
df_synthese['ID_ZH'] = id_zone_hydro

# --- VÉRIFICATION ---
print("✅ Colonne 'ID_ZH' ajoutée avec succès.")
display(df_synthese[['ZONE_PEDO', 'ID_ZH']].head())

✅ Colonne 'ID_ZH' ajoutée avec succès.


,ZONE_PEDO,ID_ZH
0,dekk/mbel_cb_avec_arbr,SSM1
1,dekk/mbel_cb_sans_arbr,SSM1
2,dekk_cb_avec_arbr,SSM1
3,dekk_cb_sans_arbr,SSM1
4,dior_cb_avec_arbr,SSM1


### 4.9. STU_DOM : Unité Typologique de Sol Dominante

La variable `STU_DOM` (Soil Typological Unit Dominante) classifie chaque `ZONE_PEDO` selon sa texture dominante.

En se basant sur la nomenclature locale, les sols sont classés comme suit :
* Les sols **Dior** sont classés comme **'sableux'**.
* Les sols **Deck** et **Deck/Mbel** sont classés comme **'argileux'**.

In [14]:
# --- 4.9 Ajout de la variable STU_DOM ---

# Créer la colonne en utilisant une condition
# np.where(condition, valeur_si_vrai, valeur_si_faux)
df_synthese['STU_DOM'] = np.where(
    df_synthese['ZONE_PEDO'].str.contains('dior'), 
    'sableux', 
    'argileux'
)

# --- VÉRIFICATION ---
print("✅ Colonne 'STU_DOM' ajoutée avec succès.")
display(df_synthese[['ZONE_PEDO', 'STU_DOM']])

✅ Colonne 'STU_DOM' ajoutée avec succès.


,ZONE_PEDO,STU_DOM
0,dekk/mbel_cb_avec_arbr,argileux
1,dekk/mbel_cb_sans_arbr,argileux
2,dekk_cb_avec_arbr,argileux
3,dekk_cb_sans_arbr,argileux
4,dior_cb_avec_arbr,sableux
5,dior_cb_sans_arbr,sableux
6,dior_cc_avec_arbr,sableux
7,dior_cc_sans_arbr,sableux


### 4.10. ID_SOL : Identifiant Unique de l'Unité de Sol

La variable `ID_SOL` est une **clé unique** qui combine plusieurs niveaux d'information pour identifier sans ambiguïté chaque unité de sol.

Elle est construite en concaténant les identifiants suivants, séparés par un tiret : `ID_ZH`, `STU_DOM`, et `ZONE_PEDO`. Cet identifiant composite assure la traçabilité et facilite les jointures dans le modèle MAELIA.

In [16]:
# --- Nettoyage de la colonne ZONE_PEDO ---
print("--- Avant le nettoyage ---")
print(df_synthese['ZONE_PEDO'].unique())

df_synthese['ZONE_PEDO'] = df_synthese['ZONE_PEDO'].str.replace('dekk/mbel', 'dekkMbel')

print("\n--- Après le nettoyage ---")
print("✅ Caractère '/' remplacé par 'dekkMbel'.")
print(df_synthese['ZONE_PEDO'].unique())


# --- 4.10 Ajout de la variable ID_SOL ---

def creer_id_sol(row):
    """
    Crée un identifiant unique en combinant plusieurs colonnes.
    """
    return f"{row['ID_ZH']}-{row['STU_DOM']}-{row['ZONE_PEDO']}"

# Appliquer la fonction pour créer la nouvelle colonne
df_synthese['ID_SOL'] = df_synthese.apply(creer_id_sol, axis=1)

# --- VÉRIFICATION ---
print("\n✅ Colonne 'ID_SOL' ajoutée avec succès.")
display(df_synthese[['ID_SOL', 'ZONE_PEDO', 'STU_DOM']])

--- Avant le nettoyage ---
['dekk/mbel_cb_avec_arbr' 'dekk/mbel_cb_sans_arbr' 'dekk_cb_avec_arbr'
 'dekk_cb_sans_arbr' 'dior_cb_avec_arbr' 'dior_cb_sans_arbr'
 'dior_cc_avec_arbr' 'dior_cc_sans_arbr']

--- Après le nettoyage ---
✅ Caractère '/' remplacé par 'dekkMbel'.
['dekkMbel_cb_avec_arbr' 'dekkMbel_cb_sans_arbr' 'dekk_cb_avec_arbr'
 'dekk_cb_sans_arbr' 'dior_cb_avec_arbr' 'dior_cb_sans_arbr'
 'dior_cc_avec_arbr' 'dior_cc_sans_arbr']

✅ Colonne 'ID_SOL' ajoutée avec succès.


,ID_SOL,ZONE_PEDO,STU_DOM
0,SSM1-argileux-dekkMbel_cb_avec_arbr,dekkMbel_cb_avec_arbr,argileux
1,SSM1-argileux-dekkMbel_cb_sans_arbr,dekkMbel_cb_sans_arbr,argileux
2,SSM1-argileux-dekk_cb_avec_arbr,dekk_cb_avec_arbr,argileux
3,SSM1-argileux-dekk_cb_sans_arbr,dekk_cb_sans_arbr,argileux
4,SSM1-sableux-dior_cb_avec_arbr,dior_cb_avec_arbr,sableux
5,SSM1-sableux-dior_cb_sans_arbr,dior_cb_sans_arbr,sableux
6,SSM1-sableux-dior_cc_avec_arbr,dior_cc_avec_arbr,sableux
7,SSM1-sableux-dior_cc_sans_arbr,dior_cc_sans_arbr,sableux


## 5. Nettoyage et Sauvegarde

In [17]:
# --- 5. Nettoyage des colonnes intermédiaires ---

# Afficher la situation avant suppression
print("--- Avant le nettoyage ---")
print(f"Nombre de colonnes : {df_synthese.shape[1]}")
#print("Colonnes :", df_synthese.columns.to_list()) # Décommenter pour voir la liste complète

# Définir les colonnes à supprimer
colonnes_a_supprimer = ['C1', 'C2', 'N1', 'N2']

# Supprimer les colonnes
df_synthese.drop(columns=colonnes_a_supprimer, inplace=True)

# Afficher la situation après suppression
print("\n--- Après le nettoyage ---")
print("✅ Colonnes intermédiaires supprimées avec succès.")
print(f"Nombre final de colonnes : {df_synthese.shape[1]}")
print("\nAperçu du DataFrame final :")
display(df_synthese.head())

--- Avant le nettoyage ---
Nombre de colonnes : 37

--- Après le nettoyage ---
✅ Colonnes intermédiaires supprimées avec succès.
Nombre final de colonnes : 33

Aperçu du DataFrame final :


,ZONE_PEDO,ARG1,ARG2,SAB1,SAB2,DAH1,DAH2,MO1,MO2,PH1,...,CSTRU,P1,P2,EG1,EG2,CAL1,CAL2,ID_ZH,STU_DOM,ID_SOL
0,dekkMbel_cb_avec_arbr,18.066667,18.800000,64.400000,64.733333,1.557333,1.514667,0.807981,0.557427,5.864444,...,0.5,30,60,0,0,0,0,SSM1,argileux,SSM1-argileux-dekkMbel_cb_avec_arbr
1,dekkMbel_cb_sans_arbr,18.100000,18.700000,64.450000,64.850000,1.556500,1.509000,0.818038,0.561162,5.868421,...,0.5,30,60,0,0,0,0,SSM1,argileux,SSM1-argileux-dekkMbel_cb_sans_arbr
2,dekk_cb_avec_arbr,17.492063,18.365079,66.047619,65.984127,1.546032,1.508889,0.753361,0.523768,5.841799,...,0.5,30,60,0,0,0,0,SSM1,argileux,SSM1-argileux-dekk_cb_avec_arbr
3,dekk_cb_sans_arbr,17.457831,18.385542,66.060241,65.951807,1.545422,1.509639,0.747136,0.521562,5.827642,...,0.5,30,60,0,0,0,0,SSM1,argileux,SSM1-argileux-dekk_cb_sans_arbr
4,dior_cb_avec_arbr,17.647668,18.481865,65.860104,65.678756,1.547565,1.507409,0.777051,0.536405,5.839062,...,0.5,30,60,0,0,0,0,SSM1,sableux,SSM1-sableux-dior_cb_avec_arbr


In [18]:
# --- Réorganisation des Colonnes ---

# 1. Définir l'ordre final des colonnes
colonnes_identifiants = ['ID_SOL', 'ID_ZH', 'STU_DOM', 'ZONE_PEDO']

colonnes_globales = ['PIRM', 'CSTRU', 'PRO']

# Ordonner les variables par type, en alternant les couches (P1 puis P2, etc.)
colonnes_par_couche_ordonnees = [
    'P1', 'P2',
    'ARG1', 'ARG2',
    'SAB1', 'SAB2',
    'DAH1', 'DAH2',
    'MO1', 'MO2',
    'PH1', 'PH2',
    'CN1', 'CN2',
    'HCC1', 'HCC2',
    'HPFP1', 'HPFP2',
    'RUPRH1', 'RUPRH2',
    'KSAT1', 'KSAT2',
    'EG1', 'EG2',
    'CAL1', 'CAL2'
]

# Combiner toutes les listes pour obtenir l'ordre final
ordre_final = colonnes_identifiants + colonnes_globales + colonnes_par_couche_ordonnees

# --- VÉRIFICATION AVANT ---
print("--- Avant la réorganisation ---")
print(f"Nombre de colonnes initial : {df_synthese.shape[1]}")
print(f"Nombre de colonnes dans la liste d'ordre : {len(ordre_final)}")
# -------------------------

# 2. Appliquer le nouvel ordre au DataFrame
df_synthese = df_synthese[ordre_final]

# --- VÉRIFICATION APRÈS ---
print("\n--- Après la réorganisation ---")
print(f"Nombre de colonnes final : {df_synthese.shape[1]}")

if df_synthese.shape[1] == len(ordre_final):
    print("\n✅ Vérification réussie : Aucune colonne n'a été perdue.")
else:
    print("\n🚨 ATTENTION : Le nombre de colonnes a changé, une colonne a peut-être été oubliée.")

print("\nAperçu du DataFrame final ordonné :")
display(df_synthese.head())

--- Avant la réorganisation ---
Nombre de colonnes initial : 33
Nombre de colonnes dans la liste d'ordre : 33

--- Après la réorganisation ---
Nombre de colonnes final : 33

✅ Vérification réussie : Aucune colonne n'a été perdue.

Aperçu du DataFrame final ordonné :


,ID_SOL,ID_ZH,STU_DOM,ZONE_PEDO,PIRM,CSTRU,PRO,P1,P2,ARG1,...,HPFP1,HPFP2,RUPRH1,RUPRH2,KSAT1,KSAT2,EG1,EG2,CAL1,CAL2
0,SSM1-argileux-dekkMbel_cb_avec_arbr,SSM1,argileux,dekkMbel_cb_avec_arbr,420.24,0.5,60,30,60,18.066667,...,9.113581,9.235677,32.438357,32.540670,322.74,242.055,0,0,0,0
1,SSM1-argileux-dekkMbel_cb_sans_arbr,SSM1,argileux,dekkMbel_cb_sans_arbr,348.00,0.5,60,30,60,18.100000,...,8.976415,9.353365,31.926052,32.738577,296.28,222.210,0,0,0,0
2,SSM1-argileux-dekk_cb_avec_arbr,SSM1,argileux,dekk_cb_avec_arbr,420.24,0.5,60,30,60,17.492063,...,8.657769,8.797480,30.489314,30.789434,322.74,242.055,0,0,0,0
3,SSM1-argileux-dekk_cb_sans_arbr,SSM1,argileux,dekk_cb_sans_arbr,348.00,0.5,60,30,60,17.457831,...,8.763053,8.867642,31.033094,31.073014,296.28,222.210,0,0,0,0
4,SSM1-sableux-dior_cb_avec_arbr,SSM1,sableux,dior_cb_avec_arbr,840.48,0.5,60,30,60,17.647668,...,8.677742,8.947725,30.514047,31.181659,674.64,505.980,0,0,0,0


In [19]:
# Sauvegarder la table de synthèse finale et enrichie
df_synthese.to_csv(output_csv_path, index=False, sep=';')

print("✅ Table de synthèse enrichie sauvegardée avec succès.")
print(f"   -> Emplacement : {output_csv_path}")

✅ Table de synthèse enrichie sauvegardée avec succès.
   -> Emplacement : C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\data\sols\csv\processed\donnees_typesDeSol_enrichies.csv
